# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [9]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
MODEL = 'gpt-4o-mini'

import base64
from io import BytesIO
from PIL import Image 
img_model = ""

audio_model = ''
openai = OpenAI()

In [10]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key :
    print("api key exist")
else:
    print("api key doesn't exist")

api key exist


In [ ]:
system_message = (
    "You are a helpful multilingual flight assistant. Your primary role is to help users search for flights between an origin and a destination. "
    "If the user provides both origin and destination, call the `search_flights` tool with the appropriate IATA airport codes. "
    "If either origin or destination is missing, ask follow-up questions to clarify. "
    "Always respond in the same language the user uses. "
    "If flight results include dates that appear to be in the past or far in the future, do not assume they are valid booking dates. "
    "Instead, mention that the provided data may represent historical or live tracking information, depending on API limitations. "
    "Clearly format departure and arrival times with timezone labels (e.g., '2025-10-12 20:30 UTC') and avoid misleading the user into thinking outdated flights are available for booking. "
    "If you do not know the answer or cannot find flights, clearly say so instead of guessing. "
    "When presenting flight results, format them in a structured and readable way, such as a table or bullet list. "
    "Keep responses concise, friendly, and professional."
)


In [ ]:
import requests

AviationStack_API_KEY = "7b5b6911118b3c1fb6d19a079f35ddee"

def search_flights(origin,destination):
   
    print(f"Tool find_flights called from {origin} to {destination}")


    # The API endpoint for real-time flights
    endpoint = "flights"

    # API parameters
    params = {
        'access_key': AviationStack_API_KEY,
        "dep_iata": origin,
        'arr_iata': destination  
    }

    # Construct the URL and make the request
    api_url = f"http://api.aviationstack.com/v1/{endpoint}"

    try:
        response = requests.get(api_url, params=params)
        response.raise_for_status()  # Check for errors
        flight_data = response.json()

        simplified = []
        for flight in flight_data.get("data", []):
            simplified.append({
                "airline": flight["airline"]["name"],
                "flight_number": flight["flight"]["iata"],
                "departure_time": flight["departure"]["scheduled"],
                "arrival_time": flight["arrival"]["scheduled"],
                "status": flight["flight_status"]
            })

        return simplified[:5]  # Return top 5 results only
    except requests.exceptions.RequestException as e:
        return f"Sorry, there was an error contacting the flight service: {e}"

# --- Example Usage ---
# We use IATA codes for airports (e.g., 'LHR' for London Heathrow)
destination = 'LHR'
origin = 'CDG'
flight_info = search_flights(destination,origin)
print(flight_info)



Tool find_flights called for CDG
[{'airline': 'Aeromexico', 'flight_number': 'AM6045', 'departure_time': '2025-10-12T17:35:00+00:00', 'arrival_time': '2025-10-12T19:55:00+00:00', 'status': 'active'}, {'airline': 'Delta Air Lines', 'flight_number': 'DL8300', 'departure_time': '2025-10-12T17:35:00+00:00', 'arrival_time': '2025-10-12T19:55:00+00:00', 'status': 'active'}, {'airline': 'Gol', 'flight_number': 'G35097', 'departure_time': '2025-10-12T17:35:00+00:00', 'arrival_time': '2025-10-12T19:55:00+00:00', 'status': 'active'}, {'airline': 'Kenya Airways', 'flight_number': 'KQ3081', 'departure_time': '2025-10-12T17:35:00+00:00', 'arrival_time': '2025-10-12T19:55:00+00:00', 'status': 'active'}, {'airline': 'Air Mauritius', 'flight_number': 'MK9417', 'departure_time': '2025-10-12T17:35:00+00:00', 'arrival_time': '2025-10-12T19:55:00+00:00', 'status': 'active'}]


In [3]:
search_flights_function = {
    "name": "search_flights",
    "description": "Search for available flights between an origin and a destination airport using IATA airport codes.",
    "parameters": {
        "type": "object",
        "properties": {
            "origin": {
                "type": "string",
                "description": "The IATA code of the departure airport (e.g., 'CDG' for Paris Charles de Gaulle)."
            },
            "destination": {
                "type": "string",
                "description": "The IATA code of the arrival airport (e.g., 'LHR' for London Heathrow)."
            }
        },
        "required": ["origin", "destination"],
        "additionalProperties": False
    }
}


In [5]:
tools = [{"type": "function", "function": search_flights_function}]

In [6]:
import json

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    
    origin = arguments.get('origin')
    destination = arguments.get('destination')

    # Call your flight function
    flights = search_flights(origin, destination)

    # Format the response (always return valid JSON as string!)
    response = {
        "role": "tool",
        "content": json.dumps({
            "origin": origin,
            "destination": destination,
            "flights": flights  # This could be a list or error string
        }),
        "tool_call_id": tool_call.id
    }
    
    return response, origin, destination


In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_response, origin, destination = handle_tool_call(tool_message)
        messages.append(tool_message)
        messages.append(tool_response)
        follow_up = openai.chat.completions.create(model=MODEL, messages=messages)
        return follow_up.choices[0].message.content

    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tool find_flights called for DXB
Tool find_flights called for JFK
Tool find_flights called for IST


: 